In [18]:
# Import the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import scipy.stats as stats
from scipy.stats import ttest_ind

In [19]:
# Load the dataset
data = pd.read_csv('data.csv')

# Print the first few rows of the dataset
data.head()

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES,PACK_SIZE,BRAND,LIFESTAGE,PREMIUM_CUSTOMER
0,2018-10-17,1,1000,1,5,Natural Chip Compny SeaSalt175g,2,6.0,175.0,Natural,YOUNG SINGLES/COUPLES,Premium
1,2019-05-14,1,1307,348,66,CCs Nacho Cheese 175g,3,6.3,175.0,CCs,MIDAGE SINGLES/COUPLES,Budget
2,2019-05-20,1,1343,383,61,Smiths Crinkle Cut Chips Chicken 170g,2,2.9,170.0,Smiths,MIDAGE SINGLES/COUPLES,Budget
3,2018-08-17,2,2373,974,69,Smiths Chip Thinly S/Cream&Onion 175g,5,15.0,175.0,Smiths,MIDAGE SINGLES/COUPLES,Budget
4,2018-08-18,2,2426,1038,108,Kettle Tortilla ChpsHny&Jlpno Chili 150g,3,13.8,150.0,Kettle,MIDAGE SINGLES/COUPLES,Budget


### Select control stores
The client has selected store numbers 77, 86 and 88 as trial stores and want
control stores to be established stores that are operational for the entire
observation period.
We would want to match trial stores to control stores that are similar to the trial
store prior to the trial period of Feb 2019 in terms of :
- Monthly overall sales revenue
- Monthly number of customers
- Monthly number of transactions per customer
Let's first create the metrics of interest and filter to stores that are present
throughout the pre-trial period.

In [20]:
# Create Monthly Metrics - YEARMONTH
data['DATE'] = pd.to_datetime(data['DATE'])
data['YEARMONTH'] = data['DATE'].dt.strftime('%Y%m').astype(int)

# Create Metrics
measureOverTime = data.groupby(['STORE_NBR', 'YEARMONTH']).agg(
    totSales=('TOT_SALES', 'sum'),
    nCustomers=('LYLTY_CARD_NBR', 'nunique'),
    nTxn=('TXN_ID', 'nunique'),
    nChips=('PROD_QTY', 'sum')
).reset_index()

# Derived metrics
measureOverTime['nTxnPerCust'] = measureOverTime['nTxn'] / measureOverTime['nCustomers']
measureOverTime['nChipsPerTxn'] = measureOverTime['nChips'] / measureOverTime['nTxn']
measureOverTime['avgPricePerUnit'] = measureOverTime['totSales'] / measureOverTime['nChips']

# Filter Pre-Trial Data
preTrial = measureOverTime[measureOverTime['YEARMONTH'] < 201902]

# Build Comparison Functions

# Correlation Function
def calculate_correlation(df, metric, trial_store):
    trial = df[df['STORE_NBR'] == trial_store][['YEARMONTH', metric]]

    results = []

    for store in df['STORE_NBR'].unique():
        compare = df[df['STORE_NBR'] == store][['YEARMONTH', metric]]

        merged = pd.merge(trial, compare, on='YEARMONTH', suffixes=('_trial', '_comp'))

        corr = merged[f'{metric}_trial'].corr(merged[f'{metric}_comp'])

        results.append({'Store1': trial_store, 'Store2': store, 'corr': corr})

    return pd.DataFrame(results)

# Magnitude Distance Function
def calculate_magnitude(df, metric, trial_store):
    trial = df[df['STORE_NBR'] == trial_store][['YEARMONTH', metric]]

    results = []

    for store in df['STORE_NBR'].unique():
        compare = df[df['STORE_NBR'] == store][['YEARMONTH', metric]]

        merged = pd.merge(trial, compare, on='YEARMONTH', suffixes=('_trial', '_comp'))

        merged['diff'] = abs(merged[f'{metric}_trial'] - merged[f'{metric}_comp'])

        # Normalize
        min_d = merged['diff'].min()
        max_d = merged['diff'].max()

        merged['score'] = 1 - (merged['diff'] - min_d) / (max_d - min_d + 1e-9)

        results.append({
            'Store1': trial_store,
            'Store2': store,
            'mag_score': merged['score'].mean()
        })

    return pd.DataFrame(results)

In [21]:

# Pick Control Store for Trial Store 77
trial_store = 77

corr_sales = calculate_correlation(preTrial, 'totSales', trial_store)
corr_customers = calculate_correlation(preTrial, 'nCustomers', trial_store)

mag_sales = calculate_magnitude(preTrial, 'totSales', trial_store)
mag_customers = calculate_magnitude(preTrial, 'nCustomers', trial_store)

# Merge scores
score = corr_sales.merge(mag_sales, on=['Store1','Store2'])
score = score.merge(corr_customers, on=['Store1','Store2'], suffixes=('_sales','_cust'))
score = score.merge(mag_customers, on=['Store1','Store2'])

# Final score
# rename merged mag_score columns then compute final score
score = score.rename(columns={'mag_score_x': 'mag_score_sales', 'mag_score_y': 'mag_score_cust'})

score['final'] = (
    0.25 * score['corr_sales'] +
    0.25 * score['mag_score_sales'] +
    0.25 * score['corr_cust'] +
    0.25 * score['mag_score_cust']
)

# Pick best control (exclude itself)
control_store = score[score['Store2'] != trial_store] \
    .sort_values('final', ascending=False) \
    .iloc[0]['Store2']

print(f"Best control store for trial store {trial_store} is store {control_store}")

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning:

Degrees of freedom <= 0 for slice

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning:

divide by zero encountered in divide

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning:

invalid value encountered in multiply

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning:

Degrees of freedom <= 0 for slice

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning:

divide by zero encountered in divide

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning:

invalid value encountered in multiply

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning:

Degrees of freedom <= 0 for slice

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning

Best control store for trial store 77 is store 233.0


In [22]:
"""
Now that we have found a control store, let's check visually if the drivers are indeed similar in the period before the trial.
We will put into consideration converting YEARMONTH to datetime for better plotting purposes
"""
# Filter data for trial and control stores
trial_data = preTrial[preTrial['STORE_NBR'] == trial_store]
control_data = preTrial[preTrial['STORE_NBR'] == control_store]

# Convert YEARMONTH to datetime
trial_data['YEARMONTH'] = pd.to_datetime(trial_data['YEARMONTH'], format='%Y%m')
control_data['YEARMONTH'] = pd.to_datetime(control_data['YEARMONTH'], format='%Y%m')

# Plot sales for trial and control stores
fig = go.Figure()
fig.add_trace(go.Scatter(x=trial_data['YEARMONTH'], y=trial_data['totSales'], mode='lines', name='Trial Store'))
fig.add_trace(go.Scatter(x=control_data['YEARMONTH'], y=control_data['totSales'], mode='lines', name='Control Store'))
fig.update_layout(title='Sales Over Time', xaxis_title='YEARMONTH', yaxis_title='totSales')
fig.show()

C:\Users\akpor\AppData\Local\Temp\ipykernel_15780\2816181329.py:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\akpor\AppData\Local\Temp\ipykernel_15780\2816181329.py:11: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [27]:
# Trial Impact Analysis
# Scale Control Store
trial_sales = preTrial[preTrial['STORE_NBR'] == trial_store]['totSales'].sum()
control_sales = preTrial[preTrial['STORE_NBR'] == control_store]['totSales'].sum()

scaling_factor = trial_sales / control_sales

# Apply scaling factor to control store's sales in the trial period
control = measureOverTime[measureOverTime['STORE_NBR'] == control_store].copy()
control['scaled_sales'] = control['totSales'] * scaling_factor

trial = measureOverTime[measureOverTime['STORE_NBR'] == trial_store]

# Compute % Difference
merged = pd.merge(
    trial[['YEARMONTH','totSales']],
    control[['YEARMONTH','scaled_sales']],
    on='YEARMONTH'
)

merged['pct_diff'] = abs(merged['totSales'] - merged['scaled_sales']) / merged['scaled_sales']

# Visualize Trial vs Scaled Control
fig = px.line(merged, x='YEARMONTH', y=['totSales', 'scaled_sales'],
              title='Trial vs Control Sales')

# Highlight trial period
fig.add_vrect(x0=201902, x1=201904,
              fillcolor="green", opacity=0.2,
              annotation_text="Trial Period")

fig.show()


In [24]:
# Let's repeat finding the control store and assessing the impact of the trial for each of the other two trial stores.
# Trial store 86
trial_store = 86

corr_sales = calculate_correlation(preTrial, 'totSales', trial_store)
corr_customers = calculate_correlation(preTrial, 'nCustomers', trial_store)

mag_sales = calculate_magnitude(preTrial, 'totSales', trial_store)
mag_customers = calculate_magnitude(preTrial, 'nCustomers', trial_store)

# Merge scores
score = corr_sales.merge(mag_sales, on=['Store1','Store2'])
score = score.merge(corr_customers, on=['Store1','Store2'], suffixes=('_sales','_cust'))
score = score.merge(mag_customers, on=['Store1','Store2'])

# Final score
# rename merged mag_score columns then compute final score
score = score.rename(columns={'mag_score_x': 'mag_score_sales', 'mag_score_y': 'mag_score_cust'})

score['final'] = (
    0.25 * score['corr_sales'] +
    0.25 * score['mag_score_sales'] +
    0.25 * score['corr_cust'] +
    0.25 * score['mag_score_cust']
)

# Pick best control (exclude itself)
control_store = score[score['Store2'] != trial_store] \
    .sort_values('final', ascending=False) \
    .iloc[0]['Store2']

print(f"Best control store for trial store {trial_store} is store {control_store}")
 


c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning:

Degrees of freedom <= 0 for slice

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning:

divide by zero encountered in divide

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning:

invalid value encountered in multiply

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning:

Degrees of freedom <= 0 for slice

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning:

divide by zero encountered in divide

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning:

invalid value encountered in multiply

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning:

Degrees of freedom <= 0 for slice

c:\Users\akpor\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning

Best control store for trial store 86 is store 155.0


In [25]:
# Statistical Significance
std_dev = merged[merged['YEARMONTH'] < 201902]['pct_diff'].std()

merged['t_value'] = merged['pct_diff'] / std_dev